In [0]:
--   WHAT IS THE GOLD LAYER?
--   ────────────────────────
--   Gold is the final layer — used for analysis and Tableau.
--   Two views are created here:
--
--   View 1 — gold_environment_monthly
--   A clean monthly summary of ocean conditions across the study region.
--   One row per H3 hex cell per month.
--   Used for: Tableau heatmaps, SST trend charts, chlorophyll maps.
--
--   View 2 — migration_environmental_joined
--   Every animal tracking observation enriched with ocean conditions.
--   Used for: species analysis, habitat stress, the main ATLAS findings.
--
--   KEY CONCEPTS
--   ─────────────
--   CTE (Common Table Expression)
--   A named temporary step inside a query using the WITH keyword.
--   Like a named step in a recipe — complete each step in order.
--
--   ROW_NUMBER() for deduplication
--   Sometimes the same H3 cell has two SST readings for the same month.
--   ROW_NUMBER() numbers them 1, 2, 3 and we keep only number 1.
--   Like giving each row a queue ticket and only calling ticket number 1.
--
--   Habitat Stress Index (0-3)
--   A score calculated for every animal observation:
--     +1 if SST is above 30 degrees C   (water too warm)
--     +1 if chlorophyll below 0.1       (not enough food)
--     +1 if microplastics above 1.0     (pollution present)
--   Score of 0 = healthy conditions
--   Score of 3 = all three stressors present at the same time

In [0]:
-- Create the Gold schema
CREATE SCHEMA IF NOT EXISTS mcphersonsharyn_gold;

In [0]:
CREATE OR REPLACE VIEW mcphersonsharyn_gold.gold_environment_monthly AS
 
-- Stage 1: Final validation checks
WITH validated AS (
SELECT
latitude,
longitude,
h3_res4,
h3_res5,
h3_res6,
CAST(sea_surface_temperature AS DOUBLE) AS sst,
CAST(chlorophyll_concentration AS DOUBLE) AS cc,
mei_value,
enso_phase,
microplastic_concentration,
pollution_severity,
CAST(year AS INTEGER) AS year,
CAST(month AS INTEGER) AS month
FROM mcphersonsharyn_silver.environmental_conditions_silver
WHERE sea_surface_temperature IS NOT NULL
AND sea_surface_temperature != -999
AND sea_surface_temperature BETWEEN -2 AND 40
AND (chlorophyll_concentration IS NULL
OR (chlorophyll_concentration != -999
AND chlorophyll_concentration > 0
AND chlorophyll_concentration < 100))
AND latitude BETWEEN -23 AND 22
AND longitude BETWEEN -73 AND 126
),

-- Stage 2: Number duplicate rows per H3 cell per month
-- We keep only row number 1 — the row with the highest SST
deduplicated AS (
SELECT *,
ROW_NUMBER() OVER (
PARTITION BY h3_res4, year, month
ORDER BY sst DESC
) AS rn
FROM validated
)
 
-- Return only the deduplicated rows
SELECT
latitude,
longitude,
h3_res4,
h3_res5,
h3_res6,
sst,
cc,
mei_value,
enso_phase,
microplastic_concentration,
pollution_severity,
year,
month
FROM deduplicated
WHERE rn = 1;
 

In [0]:
-- VALIDATION — null_sst and null_h3 must be zero
SELECT
count(*) AS total_rows,
count(*) filter(WHERE sst IS NULL) AS null_sst,
count(*) filter(WHERE cc IS NULL) AS null_cc,
count(*) filter(WHERE h3_res4 IS NULL) AS null_h3,
round(min(sst), 2) AS min_sst,
round(max(sst), 2) AS max_sst,
round(avg(sst), 2) AS avg_sst,
count(DISTINCT year) AS years_loaded,
count(DISTINCT month) AS months_loaded
FROM mcphersonsharyn_gold.gold_environment_monthly;

In [0]:
-- Check rows by year — all bookend years should appear
SELECT year, count(*) AS rows, round(avg(sst), 2) AS avg_sst
FROM mcphersonsharyn_gold.gold_environment_monthly
GROUP BY year
ORDER BY year;

In [0]:

-- Every animal tracking observation is matched to the ocean conditions at that location and time using the H3 hex cell, year and month.
--
-- A reef manta ray observed at 8.2N 118.5E in March 2016 gets matched to the SST, chlorophyll, ENSO phase and microplastic concentration for the H3 hex cell containing those coordinates in March 2016.

-- Left join - Keep all animal observations even if no environmental match exists.
-- Those rows will show NULL for SST etc — correct and expected.

In [0]:
CREATE OR REPLACE VIEW mcphersonsharyn_gold.migration_environmental_joined AS
SELECT
m.animal_id,
m.species_common,
m.species_scientific,
m.locality,
m.study_name,
m.sensor_type,
m.timestamp_utc,
m.latitude AS animal_latitude,
m.longitude AS animal_longitude,
m.h3_res4,
m.h3_res5,
m.year,
m.month,
m.season,
e.sea_surface_temperature AS sst,
e.chlorophyll_concentration AS cc,
e.enso_phase,
e.mei_value,
e.microplastic_concentration,
e.pollution_severity,
-- Habitat Stress Index (0 to 3)
(
CASE WHEN e.sea_surface_temperature > 30 THEN 1 ELSE 0 END
+ CASE WHEN e.chlorophyll_concentration < 0.1 THEN 1 ELSE 0 END
+ CASE WHEN e.microplastic_concentration > 1.0 THEN 1 ELSE 0 END
) AS habitat_stress_index,


--  SST label for Tableau
CASE
WHEN e.sea_surface_temperature > 30 THEN 'Thermal stress — above 30C'
WHEN e.sea_surface_temperature > 28 THEN 'Warm — 28 to 30C'
WHEN e.sea_surface_temperature > 25 THEN 'Moderate — 25 to 28C'
WHEN e.sea_surface_temperature IS NOT NULL THEN 'Cool — below 25C'
ELSE 'No SST data'
END AS sst_category,


--  CC label for Tableau
CASE
WHEN e.chlorophyll_concentration > 1.0 THEN 'High productivity'
WHEN e.chlorophyll_concentration > 0.1 THEN 'Moderate productivity'
WHEN e.chlorophyll_concentration IS NOT NULL THEN 'Low productivity'
ELSE 'No CC data'
END AS cc_category
FROM mcphersonsharyn_silver.migration_silver m
LEFT JOIN mcphersonsharyn_silver.environmental_conditions_silver e
ON m.h3_res4 = e.h3_res4
AND m.year = e.year
AND m.month = e.month;


In [0]:
-- VALIDATION
SELECT
species_common,
locality,
count(*) AS total_observations,
count(*) filter(WHERE sst IS NULL) AS unmatched_sst,
round(avg(sst), 2) AS avg_sst,
round(avg(cc), 3) AS avg_chlorophyll,
round(avg(habitat_stress_index), 2) AS avg_stress_index,
min(year) AS earliest_year,
max(year) AS latest_year
FROM mcphersonsharyn_gold.migration_environmental_joined
WHERE sst IS NOT NULL
GROUP BY species_common, locality
ORDER BY species_common, locality;

In [0]:
-- ENSO coverage check — every observation should have a phase
SELECT species_common, enso_phase, count(*) AS observations
FROM mcphersonsharyn_gold.migration_environmental_joined
WHERE sst IS NOT NULL
GROUP BY species_common, enso_phase
ORDER BY species_common, enso_phase;

In [0]:
-- Habitat stress distribution
SELECT species_common, habitat_stress_index, count(*) AS observations
FROM mcphersonsharyn_gold.migration_environmental_joined
WHERE habitat_stress_index IS NOT NULL
GROUP BY species_common, habitat_stress_index
ORDER BY species_common, habitat_stress_index;